# 01 · A Neuron and Backprop from Scratch

Build the engine under every neural network: one neuron, the activation functions and their slopes, a small
autograd engine (`Value`) that you check against PyTorch, and a two-layer network trained with gradients you
derive by hand. Afterwards you can explain backpropagation on a whiteboard and code it in an interview.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** [NumPy for AI engineers](../04_numpy_pandas_pytorch/01_numpy_for_ai_engineers.ipynb); [PyTorch essentials](../04_numpy_pandas_pytorch/03_pytorch_essentials.ipynb) covers the tensor/autograd API that is used here only to check our answers

## Why this matters in interviews

- "Explain backpropagation" and "implement it from scratch" are standard deep-learning screens. A scalar autograd
  engine in about 60 lines is the answer they expect.
- Activation choice (sigmoid → ReLU → GELU / SwiGLU) is how interviewers check that you know *why* deep nets train
  at all. The answer is always gradient flow.
- Vanishing and exploding gradients explain half of modern architecture: ReLU, careful initialisation,
  normalisation, residual connections, LSTM gates, gradient clipping.
- LLM questions reuse the same pieces: logits and softmax, the SwiGLU feed-forward, and "training costs about 3× a
  forward pass".

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `fo11` | What is the difference between AI, machine learning, deep learning and generative AI? |
| `fo06` | What are parameters, and does a bigger parameter count mean a better model? |
| `fo22` | What are logits, and what is the softmax doing? |
| `tr21` | What activation functions do modern transformers use, and why not ReLU? |
| `tr41` | Why does the (Llama) MLP have three Linear layers, why is it 8192 wide, and what does act_fn do? |

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from matplotlib.patches import FancyBboxPatch
from scipy.special import erf
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

random.seed(0); np.random.seed(0); torch.manual_seed(0)
rng = np.random.default_rng(0)
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]      # fixed series order for every chart
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=COLORS), "axes.grid": True, "grid.color": "#e1e0d9",
                     "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 1.8})
print(f"numpy {np.__version__} · torch {torch.__version__}")

## 1. One neuron

**In plain English:** a neuron takes some numbers in, multiplies each by a *weight* (how much it cares about that
input), adds a *bias* (how easily it fires), and pushes the total through an *activation function* that bends it:

`output = activation(w · x + b)`

The weights and the bias are the neuron's **parameters**, the knobs that training turns. A model's "parameter
count" is simply the total number of these knobs. With hand-picked weights, one sigmoid neuron already computes
logic gates:

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def neuron(x, w, b):
    return sigmoid(x @ w + b)                       # weighted sum, then the activation

X_gate = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
gates = {"AND": (np.array([10.0, 10.0]), -15.0),     # fires only when both inputs are on
         "OR":  (np.array([10.0, 10.0]), -5.0)}      # fires when either input is on
for name, (w, b) in gates.items():
    print(f"{name:3s}  inputs {X_gate.astype(int).tolist()}  ->  {np.round(neuron(X_gate, w, b), 3)}")

assert ((neuron(X_gate, *gates["AND"]) > 0.5) == [False, False, False, True]).all()
assert ((neuron(X_gate, *gates["OR"]) > 0.5) == [False, True, True, True]).all()

A single neuron splits its input space with one straight line (`w · x + b = 0`). XOR needs (0,1) and (1,0) on
one side and (0,0), (1,1) on the other, and no straight line does that. Rather than take it on faith, try 20,000
random neurons:

In [ ]:
W_try = rng.normal(0, 5, size=(20_000, 2))
b_try = rng.normal(0, 5, size=20_000)
fires = (X_gate @ W_try.T + b_try) > 0                 # [4 inputs, 20000 neurons]
xor = np.array([False, True, True, False])
best = (fires == xor[:, None]).mean(axis=0).max()
print(f"best XOR accuracy among 20,000 random single neurons: {best:.2f}")
assert best == 0.75                                      # 3 out of 4 at most, never all four

### Why the activation function is not optional

Stack two layers *without* an activation and you get one layer: `(x @ W1) @ W2 == x @ (W1 @ W2)`. Matrix
products compose into a single matrix, so extra depth buys nothing. The non-linearity between layers is what lets
a network bend its decision boundary. It is also the practical line between classic ML and deep learning: one
sigmoid neuron *is* logistic regression; many layers of neurons with non-linearities between them is deep
learning, and generative models are deep networks trained to produce new samples instead of a label.

In [ ]:
W1, W2 = rng.normal(size=(2, 8)), rng.normal(size=(8, 3))
x = rng.normal(size=(5, 2))
two_linear = (x @ W1) @ W2
one_linear = x @ (W1 @ W2)
with_relu = np.maximum(0, x @ W1) @ W2
print("two linear layers == one linear layer:        ", np.allclose(two_linear, one_linear))
print("with a ReLU in between, still one linear map? ", np.allclose(with_relu, one_linear))
assert np.allclose(two_linear, one_linear) and not np.allclose(with_relu, one_linear)

## 2. Activation functions and their derivatives

Backprop multiplies by the activation's **derivative** (its slope) at every layer, so the slope matters as much as
the shape. The five you must know:

| Activation | Formula | Where you see it |
|---|---|---|
| sigmoid | $\sigma(z) = 1/(1+e^{-z})$ | binary outputs, LSTM gates |
| tanh | $\tanh(z)$ | RNN / LSTM hidden state |
| ReLU | $\max(0, z)$ | CNNs, classic MLPs |
| GELU | $z\,\Phi(z)$, with $\Phi$ the standard normal CDF | BERT, GPT-2 |
| SiLU (Swish) | $z\,\sigma(z)$ | inside SwiGLU in Llama, Mistral, Qwen |

We write each function and its derivative by hand, then let PyTorch's autograd check every derivative.

In [ ]:
def tanh(z):  return np.tanh(z)
def relu(z):  return np.maximum(0.0, z)
def gelu(z):  return 0.5 * z * (1 + erf(z / np.sqrt(2)))           # z * Phi(z), the exact GELU
def silu(z):  return z * sigmoid(z)

def d_sigmoid(z): return sigmoid(z) * (1 - sigmoid(z))
def d_tanh(z):    return 1 - np.tanh(z) ** 2
def d_relu(z):    return (z > 0).astype(float)                      # convention: slope 0 at z = 0
def d_gelu(z):    return 0.5 * (1 + erf(z / np.sqrt(2))) + z * np.exp(-z ** 2 / 2) / np.sqrt(2 * np.pi)
def d_silu(z):    return sigmoid(z) * (1 + z * (1 - sigmoid(z)))

ACTS = {"sigmoid": (sigmoid, d_sigmoid, torch.sigmoid), "tanh": (tanh, d_tanh, torch.tanh),
        "ReLU": (relu, d_relu, torch.relu), "GELU": (gelu, d_gelu, F.gelu), "SiLU": (silu, d_silu, F.silu)}

z_t = torch.linspace(-5, 5, 1001, dtype=torch.float64, requires_grad=True)
z_np = z_t.detach().numpy()
for name, (f, df, f_torch) in ACTS.items():
    y_t = f_torch(z_t)
    (slope_t,) = torch.autograd.grad(y_t.sum(), z_t)                # autograd's derivative at every point
    assert np.allclose(f(z_np), y_t.detach().numpy(), atol=1e-12), name
    assert np.allclose(df(z_np), slope_t.numpy(), atol=1e-12), name
print("all 5 hand-written activations and derivatives match torch autograd on 1001 points")

In [ ]:
z = np.linspace(-5, 5, 1001)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for name, (f, df, _) in ACTS.items():
    ax1.plot(z, f(z), label=name)
    ax2.plot(z, df(z), label=name)
ax1.set(title="Activation functions", xlabel="input z", ylabel="output", ylim=(-1.5, 3))
ax2.set(title="Their derivatives: what backprop multiplies by", xlabel="input z", ylabel="slope", ylim=(-0.2, 1.2))
ax1.legend(); ax2.legend()
plt.tight_layout(); plt.show()

Read the right-hand panel. Sigmoid's slope never exceeds 0.25 and is nearly 0 once |z| > 4 (it *saturates*).
tanh peaks at 1 but saturates the same way. ReLU's slope is exactly 1 for every positive input, which is why it
made deep nets trainable, and exactly 0 for negative inputs (a neuron stuck there is a "dead ReLU"). GELU and SiLU
are smooth ReLUs: a small dip below zero and a small non-zero slope for slightly negative inputs, which trains a
little better in transformers. The numbers:

In [ ]:
zz = np.linspace(-8, 8, 160_001)
print(f"{'activation':10s} {'min output':>11s} {'max slope':>10s} {'slope at z=-4':>14s}")
for name, (f, df, _) in ACTS.items():
    print(f"{name:10s} {f(zz).min():11.3f} {df(zz).max():10.3f} {df(np.array(-4.0)):14.4f}")

assert math.isclose(d_sigmoid(zz).max(), 0.25, abs_tol=1e-9)        # the root of the vanishing gradient
assert math.isclose(d_tanh(zz).max(), 1.0, abs_tol=1e-9)
assert gelu(zz).min() < 0 and silu(zz).min() < 0                      # the smooth ReLUs dip below zero

### The modern transformer FFN: SwiGLU

Llama-style models do not use a plain `Linear → activation → Linear`. They use **SwiGLU**: two parallel up-projections,
one passed through SiLU and used as a *gate* on the other, then a down-projection:
`down( silu(gate(x)) * up(x) )`. That is why Llama's MLP prints three Linear layers and `act_fn: SiLU`.

Three matrices instead of two would add 50% more parameters, so the hidden width is shrunk from 4·d to about
(8/3)·d to keep the count equal. For Llama 3.2 3B, d = 3072 and (8/3)·3072 = 8192, the width you see in the printout.

In [ ]:
class SwiGLU(torch.nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.gate = torch.nn.Linear(d, hidden, bias=False)
        self.up = torch.nn.Linear(d, hidden, bias=False)
        self.down = torch.nn.Linear(hidden, d, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

d = 3072                                                          # Llama 3.2 3B model width
hidden = 8 * d // 3
classic_params = 2 * d * (4 * d)                                  # up d->4d, down 4d->d
swiglu_params = 3 * d * hidden                                    # gate, up, down
print(f"hidden width (8/3)*{d} = {hidden}")
print(f"per layer: classic 4x FFN {classic_params:,} params   SwiGLU {swiglu_params:,} params")
assert hidden == 8192 and swiglu_params == classic_params          # same budget, three matrices

small = SwiGLU(d=96, hidden=8 * 96 // 3)                          # same rules, small enough to run instantly
n_small = sum(p.numel() for p in small.parameters())
print(f"small SwiGLU: output shape {tuple(small(torch.randn(2, 96)).shape)}, {n_small:,} params")
assert n_small == 3 * 96 * 256 == 2 * 96 * (4 * 96)

## 3. Backpropagation: the chain rule, applied backwards over a graph

**In plain English:** during the forward pass every operation remembers its inputs. To get gradients, start at the
output with gradient 1 and walk backwards. Each operation takes the gradient arriving from above, multiplies it by
its own *local* slope, and **adds** the result into its inputs' gradients. A node may pass its gradient back only
after every node that used it has finished, and a reversed **topological sort** of the graph gives exactly that
order.

That is all `loss.backward()` does in PyTorch (reverse-mode automatic differentiation). Here is the whole engine for
scalars, in the style of Karpathy's *micrograd*. Part 1 is the core: a value, `+`, `*`, and `backward()`.

In [ ]:
class Value:
    """A scalar that remembers how it was made, so gradients can flow back through it."""

    def __init__(self, data, children=(), op="", label=""):
        self.data = float(data)
        self.grad = 0.0                        # d(final output) / d(this value), filled in by backward()
        self._backward = lambda: None          # pushes my grad into my children
        self._prev = tuple(children)
        self._op, self.label = op, label

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():                       # d(a+b)/da = 1 and d(a+b)/db = 1
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():                       # d(a*b)/da = b and d(a*b)/db = a
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def topo(self):
        """Every node that feeds into self, children before parents."""
        order, seen = [], set()
        def visit(v):
            if v not in seen:
                seen.add(v)
                for child in v._prev:
                    visit(child)
                order.append(v)
        visit(self)
        return order

    def backward(self):
        self.grad = 1.0                        # d(self)/d(self)
        for v in reversed(self.topo()):        # parents before children
            v._backward()

    def __repr__(self):
        return f"Value({self.label + ': ' if self.label else ''}data={self.data:.4f}, grad={self.grad:.4f})"

Part 2: more operations. Each new primitive is just *forward value + local derivative*; everything else (minus,
divide, reversed operands) is built from primitives, so it needs no backward of its own.

In [ ]:
def _pow(self, k):
    assert isinstance(k, (int, float)), "only constant exponents"
    out = Value(self.data ** k, (self,), f"**{k}")
    def _backward():                           # d(a^k)/da = k * a^(k-1)
        self.grad += k * self.data ** (k - 1) * out.grad
    out._backward = _backward
    return out

def _tanh(self):
    t = math.tanh(self.data)
    out = Value(t, (self,), "tanh")
    def _backward():                           # d tanh(a)/da = 1 - tanh(a)^2
        self.grad += (1 - t ** 2) * out.grad
    out._backward = _backward
    return out

def _relu(self):
    out = Value(max(0.0, self.data), (self,), "relu")
    def _backward():                           # slope 1 where a > 0, else 0
        self.grad += (self.data > 0) * out.grad
    out._backward = _backward
    return out

Value.__pow__, Value.tanh, Value.relu = _pow, _tanh, _relu
Value.__neg__ = lambda self: self * -1
Value.__sub__ = lambda self, other: self + (-other)
Value.__truediv__ = lambda self, other: self * other ** -1
Value.__radd__ = lambda self, other: self + other
Value.__rmul__ = lambda self, other: self * other
Value.__rsub__ = lambda self, other: (-self) + other
Value.__rtruediv__ = lambda self, other: Value(other) * self ** -1
print(Value(2.0) ** 3, "|", Value(3.0) - 1, "|", 1 / Value(4.0), "|", Value(-2.0).relu())

### A neuron as a computation graph

`o = tanh(x1·w1 + x2·w2 + b)`. Build it from `Value`s, call `o.backward()`, then draw the graph with every node's
value and gradient:

In [ ]:
x1, x2 = Value(2.0, label="x1"), Value(0.0, label="x2")
w1, w2 = Value(-3.0, label="w1"), Value(1.0, label="w2")
b = Value(6.8813735870195432, label="b")
x1w1 = x1 * w1;    x1w1.label = "x1*w1"
x2w2 = x2 * w2;    x2w2.label = "x2*w2"
s = x1w1 + x2w2;   s.label = "x1w1+x2w2"
n = s + b;         n.label = "n"
o = n.tanh();      o.label = "o"
o.backward()
for v in [x1, w1, x2, w2, b, n, o]:
    print(v)

In [ ]:
def draw_graph(root, title):
    """Leaves on the left, output on the right; each box shows value and gradient."""
    nodes = root.topo()
    dist = {root: 0}                                     # longest path from each node to the output
    for v in reversed(nodes):
        for c in v._prev:
            dist[c] = max(dist.get(c, 0), dist[v] + 1)
    depth = max(dist.values())
    columns = {}
    for v in nodes:
        columns.setdefault(depth - dist[v], []).append(v)
    pos = {v: (col * 2.3, (len(vs) - 1) / 2 - i) for col, vs in columns.items() for i, v in enumerate(vs)}
    fig, ax = plt.subplots(figsize=(11, 4.2))
    for v in nodes:
        x, y = pos[v]
        for c in v._prev:
            ax.annotate("", xy=(x - 0.85, y), xytext=(pos[c][0] + 0.85, pos[c][1]),
                        arrowprops=dict(arrowstyle="->", color="#898781", lw=1.2))
        leaf = not v._prev
        ax.add_patch(FancyBboxPatch((x - 0.85, y - 0.34), 1.7, 0.68, boxstyle="round,pad=0.04",
                                    fc="#e7f0ff" if leaf else "#fff1e8", ec="#2a78d6" if leaf else "#eb6834"))
        head = v.label if leaf else f"{v.label}  [{v._op}]"
        ax.text(x, y, f"{head}\ndata {v.data:.4f}\ngrad {v.grad:.4f}", ha="center", va="center", fontsize=8.5)
    rows = max(len(vs) for vs in columns.values())
    ax.set_xlim(-1.1, depth * 2.3 + 1.1); ax.set_ylim(-rows / 2 - 0.1, rows / 2 + 0.1)
    ax.axis("off"); ax.set_title(title)
    plt.show()

draw_graph(o, "o = tanh(x1*w1 + x2*w2 + b): forward values and backward gradients")

### The chain rule, written out

Read the graph right to left. With $o = \tanh(n)$ and $n = x_1 w_1 + x_2 w_2 + b$:

$$\frac{\partial o}{\partial n} = 1 - \tanh^2(n) = 1 - o^2 = 1 - 0.7071^2 = 0.5$$

$$\frac{\partial o}{\partial w_1} = \frac{\partial o}{\partial n}\cdot\frac{\partial n}{\partial (x_1 w_1)}\cdot\frac{\partial (x_1 w_1)}{\partial w_1} = 0.5 \cdot 1 \cdot x_1 = 0.5 \cdot 2 = 1.0$$

$$\frac{\partial o}{\partial x_1} = 0.5 \cdot w_1 = -1.5, \qquad \frac{\partial o}{\partial w_2} = 0.5 \cdot x_2 = 0, \qquad \frac{\partial o}{\partial b} = 0.5 \cdot 1 = 0.5$$

`+` passes the gradient through unchanged; `*` hands each input the *other* input's value. So `w2` gets zero
gradient because its input `x2` is 0: a weight whose input is zero cannot affect the output. Check the hand
derivation against the engine:

In [ ]:
d_n = 1 - o.data ** 2                                   # tanh'(n) = 1 - o^2
by_hand = {"b": d_n * 1, "w1": d_n * x1.data, "x1": d_n * w1.data, "w2": d_n * x2.data, "x2": d_n * w2.data}
for name, v in [("b", b), ("w1", w1), ("x1", x1), ("w2", w2), ("x2", x2)]:
    print(f"do/d{name:2s}  chain rule by hand = {by_hand[name]:+.4f}   Value.backward() = {v.grad:+.4f}")
    assert math.isclose(by_hand[name], v.grad, abs_tol=1e-12)

### Why every backward uses `+=`, and why PyTorch makes you zero gradients

If a value is used twice, it receives gradient from *both* uses (the multivariable chain rule sums over paths).
Hence `self.grad += ...`, never `=`. The flip side: PyTorch also accumulates *across* `backward()` calls, which is
why a training loop calls `optimizer.zero_grad()` every step.

In [ ]:
a = Value(3.0, label="a")
y = a * a + a                                  # a is used three times
y.backward()
print(f"Value: dy/da = {a.grad}   (by hand: 2a + 1 = {2 * 3.0 + 1})")
assert a.grad == 7.0

a_t = torch.tensor(3.0, requires_grad=True)
y_t = a_t * a_t + a_t
y_t.backward(retain_graph=True)
first = a_t.grad.item()
y_t.backward()                                 # a second backward() without zeroing
print(f"torch: grad after 1st backward = {first}, after 2nd = {a_t.grad.item()}  <- accumulated, stale")
assert (first, a_t.grad.item()) == (7.0, 14.0)

### Verify the engine against PyTorch

Because `Value` and `torch.Tensor` support the same operators (`+ - * / **`, `.relu()`, `.tanh()`), one Python
function can build the same graph in both engines. Compare gradients on a messy expression, then on 200 random
inputs:

In [ ]:
def expression(a, b):
    """Works on Value objects AND on torch tensors."""
    c = a + b
    d = a * b + b ** 3
    c = c + c + 1
    c = c + 1 + c + (-a)
    d = d + d * 2 + (b + a).relu()
    d = d + 3 * d + (b - a).relu()
    e = c - d
    f = e ** 2
    g = f / 2.0 + 10.0 / f
    return g + (a * b - 1).tanh() * b

def both_engines(a0, b0):
    a, b = Value(a0), Value(b0)
    g = expression(a, b); g.backward()
    at, bt = (torch.tensor(v, dtype=torch.float64, requires_grad=True) for v in (a0, b0))
    gt = expression(at, bt); gt.backward()
    return (g.data, a.grad, b.grad), (gt.item(), at.grad.item(), bt.grad.item())

ours, theirs = both_engines(-4.0, 2.0)
for label, u, t in zip(["g     ", "dg/da ", "dg/db "], ours, theirs):
    print(f"{label} ours {u:12.6f}   torch {t:12.6f}")

py_rng = random.Random(0)
for _ in range(200):
    ours, theirs = both_engines(py_rng.uniform(-3, 3), py_rng.uniform(-3, 3))
    assert all(math.isclose(u, t, rel_tol=1e-9, abs_tol=1e-9) for u, t in zip(ours, theirs))
print("200 random inputs: value and both gradients match torch.autograd every time")

### The whole training loop, on our own engine

Forward, loss, zero the gradients, backward, update. Here one tanh neuron learns OR (targets −1 / +1) with
nothing but `Value`. PyTorch's training loop in the [next notebook](02_training_loop_optimizers_lr_schedules.ipynb)
is these same five lines.

In [ ]:
data = [((0, 0), -1.0), ((0, 1), 1.0), ((1, 0), 1.0), ((1, 1), 1.0)]
wa, wb, bias = Value(0.3), Value(-0.5), Value(0.1)
knobs = [wa, wb, bias]
for step in range(80):
    preds = [(wa * u + wb * v + bias).tanh() for (u, v), _ in data]      # 1. forward
    loss = sum((p - t) ** 2 for p, (_, t) in zip(preds, data))          # 2. loss
    for k in knobs:                                                      # 3. zero the gradients
        k.grad = 0.0
    loss.backward()                                                      # 4. backward
    for k in knobs:                                                      # 5. update: step downhill
        k.data -= 0.2 * k.grad
    if step % 20 == 0 or step == 79:
        print(f"step {step:2d}   loss {loss.data:.4f}")
print("predictions:", [round(p.data, 2) for p in preds], "targets:", [t for _, t in data])
assert loss.data < 0.1 and all((p.data > 0) == (t > 0) for p, (_, t) in zip(preds, data))

## 4. A two-layer network from scratch in NumPy

The same idea with matrices instead of scalars: 2 inputs → 16 hidden ReLU units → 2 logits → softmax, on
`make_moons` (two interleaving half-circles that no straight line can separate). With N examples:

| Step | Formula | Shape |
|---|---|---|
| hidden pre-activation | `z1 = X @ W1 + b1` | [N, 16] |
| hidden activation | `h = relu(z1)` | [N, 16] |
| logits | `z2 = h @ W2 + b2` | [N, 2] |
| probabilities | `p = softmax(z2)` | [N, 2] |
| loss | `mean(-log p[correct class])` (cross-entropy) | scalar |

**Logits** are raw, unbounded scores, one per class. **Softmax** exponentiates and normalises them into
probabilities that sum to 1. Subtracting each row's max first changes nothing (softmax is shift-invariant) but
stops `exp` from overflowing.

The parameter count is 2·16 + 16 + 16·2 + 2 = 82. The output layer starts with tiny weights so the untrained
network is *unsure* (both classes near 50%). That gives a sanity check worth quoting in interviews: **at
initialisation the loss should be about ln(number of classes)**. If it is far higher, the model starts confidently
wrong and the first steps are wasted undoing that.

In [ ]:
X, y = make_moons(n_samples=600, noise=0.2, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
H = 16
init_params = {"W1": rng.normal(0, np.sqrt(2 / 2), size=(2, H)), "b1": np.zeros(H),   # He init: std sqrt(2/fan_in)
               "W2": rng.normal(0, 0.01, size=(H, 2)), "b2": np.zeros(2)}             # tiny: start unsure
print("parameter count:", sum(v.size for v in init_params.values()))

def forward(X, p):
    z1 = X @ p["W1"] + p["b1"]
    h = np.maximum(0, z1)
    logits = h @ p["W2"] + p["b2"]
    e = np.exp(logits - logits.max(axis=1, keepdims=True))   # stable softmax
    probs = e / e.sum(axis=1, keepdims=True)
    return z1, h, probs

def cross_entropy(probs, y):
    return -np.log(probs[np.arange(len(y)), y]).mean()

z1, h, probs = forward(X_tr, init_params)
loss0 = cross_entropy(probs, y_tr)
print(f"loss at init {loss0:.4f}   vs ln(2) = {np.log(2):.4f}, the loss of a 50/50 guess")
assert np.allclose(probs.sum(axis=1), 1.0)                  # softmax rows are probability distributions
assert abs(loss0 - np.log(2)) < 0.02                         # the sanity check passes

The backward pass, one line per chain-rule step:

| Gradient | Formula | Why |
|---|---|---|
| dz2 | `(p − onehot(y)) / N` | softmax + cross-entropy simplify to this: memorise it |
| dW2, db2 | `h.T @ dz2`, `dz2.sum(0)` | from `z2 = h @ W2 + b2` |
| dh | `dz2 @ W2.T` | send the gradient back through W2 |
| dz1 | `dh * (z1 > 0)` | ReLU passes gradient only where it was active |
| dW1, db1 | `X.T @ dz1`, `dz1.sum(0)` | from `z1 = X @ W1 + b1` |

Each layer costs two matrix products backwards (one for the weights, one for the input) versus one forwards, which
is where "the backward pass costs about 2× the forward" comes from. Before training, check the hand-derived gradients
against torch autograd on the same weights:

In [ ]:
def backward(X, y, z1, h, probs, p):
    N = len(y)
    dz2 = probs.copy()
    dz2[np.arange(N), y] -= 1
    dz2 /= N                                            # d loss / d logits = (p - onehot) / N
    grads = {"W2": h.T @ dz2, "b2": dz2.sum(axis=0)}
    dh = dz2 @ p["W2"].T
    dz1 = dh * (z1 > 0)                                 # ReLU gate
    grads["W1"], grads["b1"] = X.T @ dz1, dz1.sum(axis=0)
    return grads

grads = backward(X_tr, y_tr, z1, h, probs, init_params)

P = {k: torch.tensor(v, requires_grad=True) for k, v in init_params.items()}
logits_t = torch.relu(torch.tensor(X_tr) @ P["W1"] + P["b1"]) @ P["W2"] + P["b2"]
F.cross_entropy(logits_t, torch.tensor(y_tr)).backward()
for k in grads:
    err = np.abs(grads[k] - P[k].grad.numpy()).max()
    print(f"{k}: max |ours - torch| = {err:.1e}")
    assert err < 1e-12

In [ ]:
params = {k: v.copy() for k, v in init_params.items()}
lr, history = 0.5, []
for step in range(2000):                               # full-batch gradient descent
    z1, h, probs = forward(X_tr, params)
    history.append(cross_entropy(probs, y_tr))
    grads = backward(X_tr, y_tr, z1, h, probs, params)
    for k in params:
        params[k] -= lr * grads[k]

acc = lambda X_, y_: (forward(X_, params)[2].argmax(axis=1) == y_).mean()
print(f"loss {history[0]:.3f} -> {history[-1]:.3f}")
print(f"train accuracy {acc(X_tr, y_tr):.3f}   test accuracy {acc(X_te, y_te):.3f}")
assert history[-1] < 0.2 * history[0] and acc(X_te, y_te) > 0.93

In [ ]:
xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300),
                     np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300))
p_grid = forward(np.c_[xx.ravel(), yy.ravel()], params)[2][:, 1].reshape(xx.shape)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
ax1.plot(history)
ax1.set(title="Training loss (full-batch gradient descent)", xlabel="step", ylabel="cross-entropy")
ax2.contourf(xx, yy, p_grid, levels=20, cmap="coolwarm", alpha=0.55)
ax2.contour(xx, yy, p_grid, levels=[0.5], colors="#0b0b0b", linewidths=1.5)
ax2.scatter(X_te[:, 0], X_te[:, 1], c=y_te, cmap="coolwarm", edgecolor="#0b0b0b", s=18, linewidths=0.5)
ax2.set(title="Decision boundary, test points (colour = P(class 1))", xlabel="x0", ylabel="x1")
ax2.grid(False)
plt.tight_layout(); plt.show()

Two non-linear layers, 82 parameters and 2,000 hand-derived gradient steps bend a boundary between the moons. A single
neuron could only have drawn a straight line here.

## 5. Vanishing gradients: why deep sigmoid nets stop learning

The gradient that reaches layer 1 is a **product** with one factor per layer above it: (weights) × (activation
slope). Sigmoid's slope is at most 0.25, so twenty layers multiply the signal by something on the order of
0.25²⁰ ≈ 10⁻¹². The first layers receive almost no signal and stop learning. ReLU's slope is exactly 1 where it is
active, so with a suitable initialisation the product stays near 1.

The experiment: a 20-layer, 64-wide MLP, one backward pass on random data, and the gradient norm of each layer's
weights. Each net gets the initialisation designed for its activation (Xavier for sigmoid and tanh, He/Kaiming for
ReLU), so any difference comes from the activation, not from a bad init.

In [ ]:
def grad_norms_per_layer(act_cls, depth=20, width=64, matched_init=True):
    torch.manual_seed(0)
    layers = []
    for _ in range(depth):
        lin = torch.nn.Linear(width, width)
        if matched_init:
            if act_cls is torch.nn.ReLU:
                torch.nn.init.kaiming_normal_(lin.weight, nonlinearity="relu")
            else:
                torch.nn.init.xavier_normal_(lin.weight)
            torch.nn.init.zeros_(lin.bias)
        layers += [lin, act_cls()]
    net = torch.nn.Sequential(*layers, torch.nn.Linear(width, 1))
    x, target = torch.randn(256, width), torch.randn(256, 1)
    F.mse_loss(net(x), target).backward()
    return np.array([m.weight.grad.norm().item() for m in net if isinstance(m, torch.nn.Linear)][:-1])

norms = {act.__name__: grad_norms_per_layer(act) for act in [torch.nn.Sigmoid, torch.nn.Tanh, torch.nn.ReLU]}
fig, ax = plt.subplots(figsize=(10, 4))
for name, g in norms.items():
    ax.semilogy(range(1, len(g) + 1), g, marker="o", markersize=4, label=name)
ax.set(title="Gradient norm per layer in a 20-layer MLP (layer 1 = next to the input)",
       xlabel="layer", ylabel="||dL/dW|| (log scale)", xticks=range(1, 21))
ax.legend(); plt.show()
for name, g in norms.items():
    print(f"{name:8s} layer-1 grad {g[0]:.1e}   layer-20 grad {g[-1]:.1e}   ratio {g[0] / g[-1]:.1e}")
assert norms["Sigmoid"][0] / norms["Sigmoid"][-1] < 1e-9           # vanished: 9+ orders of magnitude
assert 0.1 < norms["ReLU"][0] / norms["ReLU"][-1] < 10              # ReLU + He init: flows fine

The sigmoid net's first layer gets a gradient roughly a trillion times smaller than its last layer: it is frozen.
tanh with Xavier init and ReLU with He init keep every layer within a small factor of the others.

Every modern trick is a way of keeping that product near 1:

| Problem | Fix | Where it is shown |
|---|---|---|
| activation slope < 1 shrinks the signal | ReLU / GELU / SiLU in hidden layers | this notebook |
| weights too small or too large | variance-preserving init (Xavier for tanh, He for ReLU) | this notebook, exercise 4 |
| activations drift in scale with depth | LayerNorm / RMSNorm / BatchNorm | [03 · regularization and normalization](03_regularization_dropout_batchnorm.ipynb) |
| a long chain of multiplications | residual connections `x + f(x)`: a gradient highway | [03](03_regularization_dropout_batchnorm.ipynb) |
| the same, through time in an RNN | LSTM / GRU gates, then attention | [05 · embeddings, RNNs, attention](05_embeddings_rnn_to_attention.ipynb) |
| the opposite: exploding gradients | gradient clipping | [02 · training loop](02_training_loop_optimizers_lr_schedules.ipynb) |

## Try it yourself

**1.** Add an `exp()` operation to `Value` (its local derivative is itself), build `sigmoid(v) = 1 / (1 + exp(-v))`
from it, and check the gradient against `torch.sigmoid`.

In [ ]:
# Solution — try it yourself first, then run this
def _exp(self):
    out = Value(math.exp(self.data), (self,), "exp")
    def _backward():                                   # d e^a / da = e^a
        self.grad += out.data * out.grad
    out._backward = _backward
    return out

Value.exp = _exp
v = Value(0.7)
sig = 1 / (1 + (-v).exp())
sig.backward()
v_t = torch.tensor(0.7, dtype=torch.float64, requires_grad=True)
torch.sigmoid(v_t).backward()
print(f"sigmoid(0.7) = {sig.data:.6f}   grad ours {v.grad:.6f}   torch {v_t.grad.item():.6f}")
assert math.isclose(v.grad, v_t.grad.item(), rel_tol=1e-12)

**2.** Swap the NumPy network's hidden ReLU for tanh. What changes in `forward` and in `backward`? (Hint: tanh's
derivative can be written using its own output `h`.) Train it and compare the test accuracy.

In [ ]:
# Solution — try it yourself first, then run this
def forward_tanh(X, p):
    z1 = X @ p["W1"] + p["b1"]
    h = np.tanh(z1)
    logits = h @ p["W2"] + p["b2"]
    e = np.exp(logits - logits.max(axis=1, keepdims=True))
    return z1, h, e / e.sum(axis=1, keepdims=True)

def backward_tanh(X, y, z1, h, probs, p):
    dz2 = probs.copy(); dz2[np.arange(len(y)), y] -= 1; dz2 /= len(y)
    dz1 = (dz2 @ p["W2"].T) * (1 - h ** 2)              # the only line that changes: tanh' = 1 - h^2
    return {"W2": h.T @ dz2, "b2": dz2.sum(0), "W1": X.T @ dz1, "b1": dz1.sum(0)}

pt = {k: v.copy() for k, v in init_params.items()}
for step in range(2000):
    z1_, h_, probs_ = forward_tanh(X_tr, pt)
    g_ = backward_tanh(X_tr, y_tr, z1_, h_, probs_, pt)
    for k in pt:
        pt[k] -= 0.5 * g_[k]
acc_tanh = (forward_tanh(X_te, pt)[2].argmax(1) == y_te).mean()
print(f"tanh hidden layer: test accuracy {acc_tanh:.3f}   (ReLU version: {acc(X_te, y_te):.3f})")
assert acc_tanh > 0.9

**3.** Gradient checking: estimate `dLoss/dw` for a few individual weights with central finite differences,
`(L(w + ε) − L(w − ε)) / 2ε`, and compare with `backward()`. This is the standard way to debug a hand-written
backward pass.

In [ ]:
# Solution — try it yourself first, then run this
eps = 1e-6
p0 = {k: v.copy() for k, v in init_params.items()}
analytic = backward(X_tr, y_tr, *forward(X_tr, p0), p0)
for k, idx in [("W1", (0, 3)), ("W1", (1, 7)), ("b1", (5,)), ("W2", (4, 1)), ("b2", (0,))]:
    plus = {n: v.copy() for n, v in p0.items()};  plus[k][idx] += eps
    minus = {n: v.copy() for n, v in p0.items()}; minus[k][idx] -= eps
    numeric = (cross_entropy(forward(X_tr, plus)[2], y_tr) - cross_entropy(forward(X_tr, minus)[2], y_tr)) / (2 * eps)
    rel = abs(numeric - analytic[k][idx]) / max(1e-12, abs(numeric) + abs(analytic[k][idx]))
    print(f"{k}{list(idx)}: numeric {numeric:+.6f}   analytic {analytic[k][idx]:+.6f}   rel. error {rel:.1e}")
    assert rel < 1e-5

**4.** Predict first: if you rerun the 20-layer experiment with PyTorch's *default* `nn.Linear` initialisation
(weights uniform in ±1/√fan_in, variance 1/(3·fan_in)) instead of the matched ones, does the ReLU net still keep its
gradients?

In [ ]:
# Solution — try it yourself first, then run this
for act in [torch.nn.Sigmoid, torch.nn.Tanh, torch.nn.ReLU]:
    g = grad_norms_per_layer(act, matched_init=False)
    print(f"{act.__name__:8s} default init: layer-1 / layer-20 gradient ratio = {g[0] / g[-1]:.1e}")
g_relu_default = grad_norms_per_layer(torch.nn.ReLU, matched_init=False)
assert g_relu_default[0] / g_relu_default[-1] < 1e-3
print("No: with variance 1/(3*fan_in), each ReLU layer shrinks the signal (He init uses 2/fan_in to cancel")
print("ReLU zeroing half its inputs). Init matters as much as the activation; normalisation and residuals")
print("are what make real deep nets robust to it.")

## Say it in an interview

- **Backprop in 30 seconds:** "It is reverse-mode automatic differentiation. The forward pass computes and stores
  every intermediate value; the backward pass walks the computation graph in reverse topological order, and each
  node multiplies the incoming gradient by its local derivative and *adds* it into its inputs' gradients. One
  backward pass gives the gradient of a scalar loss with respect to every parameter, for roughly twice the cost of
  the forward pass."
- **Key numbers:** backward ≈ 2× forward FLOPs, so a training step ≈ 3× a forward pass (≈ 6 × parameters FLOPs per
  training token for an LLM). Sigmoid's maximum slope is 0.25. The gradient of softmax + cross-entropy with respect
  to the logits is `p − onehot(y)`.
- **Why non-linearities:** without them any depth collapses to one linear map (asserted above).
- **Why not sigmoid / tanh in hidden layers:** they saturate, so gradients vanish with depth. ReLU has slope 1 but
  can die; GELU and SiLU are smooth ReLUs; Llama-style FFNs use SwiGLU (three matrices, hidden ≈ 8/3·d so the
  parameter count matches a 4·d FFN).
- **Parameters** are the learned weights and biases; the count sets memory (2 bytes each in bf16), but more
  parameters is not automatically a better model: data, training and architecture matter as much.
- **Follow-ups they ask:** "Why accumulate gradients?" (a value used twice gets both contributions; and it is why
  you must `zero_grad()`). "What does backward need to store?" (the forward activations: that is activation
  memory, and gradient checkpointing trades recomputation for it). "How do you test a hand-written backward?"
  (finite-difference gradient check).
- **Traps:** computing softmax then `log` yourself (use `F.cross_entropy` on raw logits, which uses the stable
  log-sum-exp); forgetting `zero_grad()`; blaming the activation when the init is wrong (exercise 4).

## Next

- [02 · Training loop, optimizers and LR schedules](02_training_loop_optimizers_lr_schedules.ipynb): the same five
  lines in PyTorch, then everything that makes them work at scale.
- [03 · Regularization, dropout and normalization](03_regularization_dropout_batchnorm.ipynb): residual connections
  and normalisation, the other half of the vanishing-gradient fix.
- [PyTorch essentials](../04_numpy_pandas_pytorch/03_pytorch_essentials.ipynb) ·
  [Linear regression and gradient descent](../05_machine_learning/02_linear_regression_and_gradient_descent.ipynb) ·
  [Transformer block and tiny GPT](../07_genai_foundations/04_transformer_block_and_tiny_gpt.ipynb)
- Theory: [dl_fundamentals course](../../dl_fundamentals/index.html) (chapters 1–5 and 8) ·
  [interview bank](../../ai_interview_prep/index.html)